# create_one_to_many_sql.ipynb

An example of setting up a SQLite3 database with containing a one_to_many relationship between a parent and child tables.
Both tables have a unique id field for each record.
The biggest problem I had was setting up try ... except blocks to trap database integrity errors which caused crashes.

In [ ]:
import sqlite3
import pandas as pd
from icecream import ic

In [ ]:
def create_database(db_path: str):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS people (
            person_id INTEGER PRIMARY KEY AUTOINCREMENT,
            Name TEXT UNIQUE,
            Age INTEGER
        );
    """)
    conn.commit()
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS notes (
            note_id INTEGER PRIMARY KEY AUTOINCREMENT,
            person_id INTEGER,
            note TEXT,
            FOREIGN KEY(person_id) REFERENCES people(person_id) ON DELETE CASCADE  
        );
    """)
    conn.commit()
    
    conn.close()
    
# # Example usage:

# create_database('my_database.sqlite3')  

In [ ]:
db_path = 'my_database.sqlite3'

people_data = [
    {'Name': 'Aubrey Moore', 'Age': 74},
    {'Name': 'Jane Ginlo Moore', 'Age': 72}
]

In [ ]:
# Connect to SQLite database (database and tables are created one if it do not exist)
# Database tables are not modified if they already exist

create_database(db_path)

people_df = pd.DataFrame(people_data)

conn = sqlite3.connect(db_path)

print(f'dataframe before populating people table in {db_path}')
print(people_df)

try:
    people_df.to_sql(name='people', con=conn, if_exists='append', index=False)
    conn.commit()
    print(f"Successfully populated people table with data in dataframe.")

except sqlite3.IntegrityError as e:
    # Catch the specific error
    print(f"Caught an integrity error for user ID when populating people table: {e}")
    # Roll back the failed transaction
    conn.rollback()
    print("Transaction rolled back. Continuing with next record.")

except Exception as e:
    # Catch any other potential errors
    print(f"An unexpected error occurred: {e}")
    print('recovering from error')
    conn.rollback()

people_df = pd.read_sql('SELECT * FROM people;', con=conn)
print(f'dataframe after populating people table in {db_path}')
print(people_df)

conn.close()


dataframe before populating people table in my_database.sqlite3
               Name  Age
0      Aubrey Moore   74
1  Jane Ginlo Moore   72
An unexpected error occurred: Execution failed
recovering from error
dataframe after populating people table in my_database.sqlite3
   person_id              Name  Age
0          1      Aubrey Moore   74
1          2  Jane Ginlo Moore   72
